In [42]:
# imports:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

Complete ETL Example

In [43]:
# Create SparkSession:
spark = SparkSession.builder.appName('ETL Example').getOrCreate()

In [44]:
df = spark.read.csv(
    'dataFiles/employees.csv',
    header=True,
    inferSchema=True
)

df.show()

+---+-----+----------+------+----+
| id| name|department|salary| age|
+---+-----+----------+------+----+
|  1| NULL|        IT| 60000|NULL|
|  2|Alice|      NULL| 50000|  28|
|  3|  Bob|        IT| 70000|  35|
|  4| Emma|   Finance| 80000|  40|
|  5| NULL|     Sales| 55000|  29|
|  6|David|      NULL| 75000|  32|
|  3|  Bob|        IT| 70000|  35|
+---+-----+----------+------+----+



In [45]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [46]:
df = df.na.fill(
    {
        'name': 'Unkown',
        'age': 18,
        'department': 'Other'
    }
)

df.show()

+---+------+----------+------+---+
| id|  name|department|salary|age|
+---+------+----------+------+---+
|  1|Unkown|        IT| 60000| 18|
|  2| Alice|     Other| 50000| 28|
|  3|   Bob|        IT| 70000| 35|
|  4|  Emma|   Finance| 80000| 40|
|  5|Unkown|     Sales| 55000| 29|
|  6| David|     Other| 75000| 32|
|  3|   Bob|        IT| 70000| 35|
+---+------+----------+------+---+



In [47]:
# filter
df = df.filter(
    col('salary') > 60000
)

df.show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  3|  Bob|        IT| 70000| 35|
|  4| Emma|   Finance| 80000| 40|
|  6|David|     Other| 75000| 32|
|  3|  Bob|        IT| 70000| 35|
+---+-----+----------+------+---+



In [48]:
# add derived column
df = df.withColumn(
    'bonus',
    col('salary') * 0.10
 )

df.show()

+---+-----+----------+------+---+------+
| id| name|department|salary|age| bonus|
+---+-----+----------+------+---+------+
|  3|  Bob|        IT| 70000| 35|7000.0|
|  4| Emma|   Finance| 80000| 40|8000.0|
|  6|David|     Other| 75000| 32|7500.0|
|  3|  Bob|        IT| 70000| 35|7000.0|
+---+-----+----------+------+---+------+



In [49]:
summary_df = df.groupBy(
    'department'
).agg(
    avg('salary').alias("avg_salary"),
    sum('bonus').alias('total_bonus'),
    count('*').alias('total_employees')
)

In [50]:
# Save Results:

summary_df.write.mode('overwrite').parquet("dataFiles/emp_dept_wise_summary")


Transformations:
    Common Transformations used in Production

Filtering ==> Active Customers only
Deduplication ==> Remove Duplicate transactions
Data Cleaning ==> Null Handelling
standerdization ==> Convert names to uppercase
Enrichment ==> Join customer master data
Aggregation ==> Daily salaes total (KPI)
Window Functions ==> Rank Top Customers
Date Transformation ==> Extract Year/ Month/Day
Pivoting ==> Monthly Sales by Region
Parsing JSON ==> Nested API Data
Incremental Processing ==> Process only new data